In [ ]:
from datetime import datetime

etherpad_url = "http://localhost:9001/"
marker_base = f"zq{datetime.now().strftime('%Y%m%d%H%M%S')}x"
update_iterations = 3
large_body_chars = 2000000
default_result_path = None
close_on_fail = False
transition_timeout = 10000

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# ep_search E2E Test - Update Ordering

Test that rapid successive updates of a pad leave the search index at the
latest content. Importing an HTML file rewrites the pad as two immediate
revisions (clear, then insert), so the index updates for both revisions race;
the index must converge to the final revision.

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## Step 1: Create a pad and wait until it is indexed

In [ ]:
import asyncio
import time

pad_id = f"test-ordering-{int(time.time())}"


async def wait_for_search_hit(page, query, timeout_ms):
    deadline = time.monotonic() + timeout_ms / 1000
    result = None
    while time.monotonic() < deadline:
        response = await page.request.get(f"{etherpad_url}search?query={query}")
        assert response.ok, response.status
        result = await response.json()
        if result["numFound"] == 1:
            return result
        await asyncio.sleep(1)
    raise AssertionError(f"Pad not found by query {query}: {result}")


async def _step(page):
    await page.goto(f"{etherpad_url}p/{pad_id}")

    await expect(page.locator('iframe[name="ace_outer"]')).to_be_visible(timeout=transition_timeout)
    iframe_locator = page.frame_locator('iframe[name="ace_outer"]').frame_locator('iframe[name="ace_inner"]')
    editor = iframe_locator.locator('#innerdocbody')
    await expect(editor).to_be_visible(timeout=transition_timeout)

    await editor.click()
    await editor.press_sequentially(f"OrderingTest {marker_base}0000")

    await wait_for_search_hit(page, f"{marker_base}0000", transition_timeout)

await run_pw(_step)

## Step 2: Rewrite the pad with a large body, then immediately with a small one

Each import rewrites the pad, and the index update for a large body takes
longer than the one for a small body that follows right after it. The index
updates must be applied in the pad event order; if the slow update for the
older revision lands last, the index keeps the older content and the search
never converges to the last marker.

In [ ]:
filler = "検索順序の検証のための長い本文です。" * (large_body_chars // 18)

async def import_html(page, body):
    response = await page.request.post(
        f"{etherpad_url}p/{pad_id}/import",
        multipart={
            "file": {
                "name": "import.html",
                "mimeType": "text/html",
                "buffer": f"<html><body>{body}</body></html>".encode(),
            },
        },
    )
    assert response.ok, (response.status, await response.text())


async def _step(page):
    for i in range(1, update_iterations + 1):
        large_marker = f"{marker_base}{i:04d}L"
        last_marker = f"{marker_base}{i:04d}S"
        await import_html(page, f"<p>OrderingTest</p><p>{large_marker}</p><p>{filler}</p>")
        await import_html(page, f"<p>OrderingTest</p><p>{last_marker}</p>")
        result = await wait_for_search_hit(page, last_marker, transition_timeout)
        print(i, last_marker, result["numFound"])

await run_pw(_step)

## Cleanup

In [ ]:
await finish_pw_context(screenshot=True, last_path=default_result_path)

In [ ]:
!rm -rf {work_dir}